In [ ]:
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import GRB

In [ ]:
def coordinate(i,j,GridNumber):
    x = 1/(GridNumber)*(i-1)
    y = 1/(GridNumber)*(j-1)
    return x, y

def coordinate_10(i,j):
    x = 0.1*(i-1)
    y = 0.1*(j-1)
    return x, y

def coordinate_20(i,j):
    x = 0.05*(i-1)
    y = 0.05*(j-1)
    return x, y

def coordinate_step(i,j,GridNumber):
    x = i
    y = j
    return x, y


In [ ]:
## Construct the matrix

demand_grid_number = 20
hub_grid_number = 20
Totalofpair = (pow(demand_grid_number+1,2)+1)*pow(demand_grid_number+1,2)/2
print("The total number of pair is:", Totalofpair)


ColumnsName=[f'P{i,j}' for i in range(1, hub_grid_number + 2) for j in range(1, hub_grid_number + 2)]
# print(ColumnsName)

Rowindex = []

Dia_index = []
Dia_counter = 0

for i in range(1, demand_grid_number + 2):
    for j in range(1, demand_grid_number + 2):
        for l in range(j,demand_grid_number+2):
            if l == j:
                Dia_index.append(Dia_counter)

            temp = f'O{i,j}_D{i,l}'
            Rowindex.append(temp)
            Dia_counter = Dia_counter+1

        for k in range(i+1,demand_grid_number + 2):
            for l in range(1,demand_grid_number + 2):
                temp = f'O{i,j}_D{k,l}'
                Rowindex.append(temp)
                Dia_counter = Dia_counter+1
        
print("the true row index is ",len(Rowindex))

df = pd.DataFrame( index= Rowindex, columns=ColumnsName)

for i in range(1, demand_grid_number + 2):
    for j in range(1, demand_grid_number + 2):
        for l in range(j,demand_grid_number+2):
        
            for m in range(1, hub_grid_number + 2):
                for n in range(1, hub_grid_number + 2):

                    Oi,Oj = coordinate(i,j,demand_grid_number)
                    Di,Dj = coordinate(i,l,demand_grid_number)
                    Pi,Pj = coordinate(m,n,hub_grid_number)
                    
                
                    #df.loc[f'O{i,j}_D{i,l}',f"P{m,n}"] = np.abs(Oi-Pi)+np.abs(Oj-Pj)+np.abs(Di-Pi)+np.abs(Dj-Pj)

                    df.loc[f'O{i,j}_D{i,l}',f"P{m,n}"] = round((np.abs(Oi-Pi)+np.abs(Oj-Pj)+np.abs(Di-Pi)+np.abs(Dj-Pj))*100)

        for k in range(i+1,demand_grid_number + 2):
            for l in range(1,demand_grid_number + 2):

                for m in range(1, hub_grid_number + 2):
                    for n in range(1, hub_grid_number + 2):

                        Oi,Oj = coordinate(i,j,demand_grid_number)
                        Di,Dj = coordinate(k,l,demand_grid_number)
                        Pi,Pj = coordinate(m,n,hub_grid_number)

                        #df.loc[f'O{i,j}_D{k,l}',f"P{m,n}"] = np.abs(Oi-Pi)+np.abs(Oj-Pj)+np.abs(Di-Pi)+np.abs(Dj-Pj)
                        df.loc[f'O{i,j}_D{k,l}',f"P{m,n}"] = round((np.abs(Oi-Pi)+np.abs(Oj-Pj)+np.abs(Di-Pi)+np.abs(Dj-Pj))*100)


print(df)



In [ ]:
## Calculate distance d_ik

m = gp.Model()
Pnumber = 30
m.Params.MIPGap = 0
#Gap=m.setParam('MIPGap', 0)
# variables: t_ij, y_j, 

Pair_range = range(len(df))
Location_range = range(len(df.columns))




t = {}
for i in Pair_range:
    for k in Location_range:
        t[i,k] = m.addVar(vtype=GRB.CONTINUOUS,name = f"t_+{i,k}")

y = m.addVars(Location_range ,vtype=GRB.BINARY, name="y_i")

Non_Dia_index = [item for item in Pair_range if item not in Dia_index]
# Obejective function
obj = gp.quicksum(
        
        df.iloc[i,j] * t[i,j] 
        for i in Pair_range
        for j in Location_range
    ) + gp.quicksum(
        
        df.iloc[i,j] * t[i,j] 
        for i in Non_Dia_index
        for j in Location_range
    )

# Constraints

for i in Pair_range:
    m.addConstr(gp.quicksum(t[i,k] for k in Location_range) == 1,name = f"constraint1_{i}")


m.addConstr(gp.quicksum(y[j] for j in Location_range) == Pnumber,name = "plocation")

for i in Pair_range:
    for k in Location_range:
        m.addConstr(t[i,k] <= y[k],name = f"constraint2_{i,k}")


print("start the optimization")

m.setObjective(obj, GRB.MINIMIZE)
m.optimize()
runtime = m.Runtime
print("running time is ", runtime)

hubs = []
for k,y in y.items():
    if y.x > 0:
        hubs.append(k)

for i in hubs:
     print(ColumnsName[i])